# Inside the winning region
### Environment × safety monitor → deterministic safety shield

Watch the greatest fixed point emerge **one state at a time**. Inspect each
candidate action, follow the removal cascade, and rewind any decision.

From the repository root, install the project in this notebook's Python environment:
```sh
python -m pip install -e .
```
The GUI uses an offline browser window, native SVG and your installed Source Sans
Pro / Source Sans 3 fonts. No server or additional GUI package is needed.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / "notebooks/tools/winning-region/explorer.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Start Jupyter inside the MASA-Safe-RL checkout.")
TOOL = REPO / "notebooks/tools/winning-region"
if str(TOOL) not in sys.path:
    sys.path.insert(0, str(TOOL))

from explorer import build_explorer
from masa.common.ltl import Atom, DFA
from masa.envs.tabular.frozen_lake import FrozenLake

## 1 · Choose an environment and a safety property

This compact, slippery lake exposes all positive-probability outcomes. Change
`desc` or `is_slippery` to compare the resulting winning regions. Replace `env`
with any raw finite MASA environment exposing an exact transition model and
`label_fn(state)`.

The property is **G ¬hole**: never visit a hole. MASA uses a **bad-prefix DFA**,
so accepting state `1` means a violation. Missing guards imply a self-loop.
Construct other safety properties with `DFA`, `Atom`, `And`, `Or`, `Neg`, etc.;
`property_name` below is a display title, not an LTL string parser.

In [ ]:
env = FrozenLake(
    desc=["SFF", "FHF", "FFG"],
    is_slippery=True,
)

dfa = DFA(states=[0, 1], initial=0, accepting=[1])
dfa.add_edge(0, 1, Atom("hole"))

PROPERTY_NAME = "G ¬hole · always avoid holes"
OPEN_IN_BROWSER = True  # Set False for a remote kernel or an inline GUI.

## 2 · Open the explorer

A blue outline follows the current state. **Amber is only a proposed removal**;
all decisions in a round read the same old set. Removals are committed together.

Use Play, Back, Step, Next round, the speed selector or the timeline. Click a
node to inspect actions; hover an action to highlight its complete support.
Drag nodes to arrange the graph, drag empty space to pan, and scroll to zoom.

Real action names come from `env.action_names()` when available. Both final masks
are verified against MASA's production solver before the GUI opens.

In [ ]:
view = build_explorer(
    env,
    dfa,
    property_name=PROPERTY_NAME,
    # action_names={0: "Left", 1: "Down", 2: "Right", 3: "Up"},
    # state_names={0: "Start", 8: "Goal"},
    max_states=500,
)

if OPEN_IN_BROWSER:
    html_path = view.open()
    print(f"GUI document: {html_path}")
else:
    view.show(height=950)

## 3 · Work with the synthesized shield

A product ID encodes `q_index * n_states + s`. The returned masks are read-only.
The full product includes combinations unreachable from reset: consult the
inspector's reset mapping when interpreting a base state.

Safety is infinite-horizon and does **not** require eventually reaching a goal.
A deterministic safety guarantee can still permit several actions and account
for stochastic environment transitions.

In [ ]:
import numpy as np

winning_product_ids = np.flatnonzero(view.trace.winning)
allowed_action_mask = view.trace.allowed
print("Winning product IDs:", winning_product_ids.tolist())
print("Allowed-action mask shape:", allowed_action_mask.shape)

# Optional: export a portable, offline GUI.
# view.save(REPO / "winning-region.html")
# SVG diagrams and the full trace JSON can also be exported from the GUI.

The explorer keeps a model snapshot, not a live environment reference.
Closing the environment does not close the exported GUI. Re-run the configuration
and build cells after changing the environment or safety property.

In [ ]:
env.close()